Current State - ResNet50 [batch,512], DistilBERT[batch,768]
Now we concatenate so the model sees both Image and texts simultaneously(multimodal)

In [3]:
#load weights from previous files
import torch
import torch.nn as nn
from transformers import DistilBertTokenizer, DistilBertModel
import numpy as np
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt
import torch.nn.functional as F
from sklearn.model_selection import train_test_split

In [4]:
#loading image embedder
class ImageEmbedder(nn.Module):
  def __init__(self):
    super().__init__()

    resnet = models.resnet50(pretrained = True)
    for param in resnet.parameters():
      param.requires_grad=False
    self.backbone = nn.Sequential(*list(resnet.children())[:-1])
    self.embedding_head = nn.Sequential(
        nn.Flatten(),
        nn.Linear(2048,512),
        nn.ReLU()
    )

  def forward(self,x):
    return self.embedding_head(self.backbone(x))




In [5]:
#loading text embedder
class TextEmbedder(nn.Module):
  def __init__(self):
    super().__init__()

    self.bert = DistilBertModel.from_pretrained('distilbert-base-uncased')
    for param in self.bert.parameters():
      param.requires_grad = False

    self.embedding_head = nn.Sequential(
        nn.Linear(768,768),
        nn.ReLU(),
        nn.Dropout(0.1)
    )

  def forward(self,input_ids, attention_mask):
    outputs = self.bert(input_ids = input_ids, attention_mask = attention_mask)
    cls_token = outputs.last_hidden_state[:,0,:]
    return self.embedding_head(cls_token)




In [11]:
#loading both embedders
image_embedder = ImageEmbedder()
text_embedder = TextEmbedder()

image_embedder.eval()
text_embedder.eval()

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


TextEmbedder(
  (bert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1): Linear(in

In [12]:
#building the fusion model
class MultimodalAdScorer(nn.Module):
  def __init__(self):
    super().__init__()

    #both embedders live in the model
    self.image_embedder = image_embedder
    self.text_embedder = text_embedder

    #fusion layer (text - 768) + (image - 512) = 1280 -> 512 -> 256 -> 1

    self.fusion = nn.Sequential(
      nn.Linear(1280,512),
      nn.ReLU(),
      nn.Dropout(0.3),
      nn.Linear(512,256),
      nn.ReLU(),
      nn.Dropout(0.3),
      nn.Linear(256, 1)
    )

  def forward(self, image_tensor, input_ids, attention_mask):
    img_emb = self.image_embedder(image_tensor)
    text_emb = self.text_embedder(input_ids, attention_mask)

    #concat both branches
    combined = torch.cat([img_emb, text_emb], dim = 1)

    score = self.fusion(combined) #running the fusion layer to produce a score

model = MultimodalAdScorer()
print(model)


MultimodalAdScorer(
  (image_embedder): ImageEmbedder(
    (backbone): Sequential(
      (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU(inplace=True)
      (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (4): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (relu): ReLU